# Video to 3D Scene Restructured
This notebook allows you to train a 3D scene using Gaussian Splatting from either a test dataset or your own video.

In [ ]:
!nvidia-smi
!python -m pip install -q -U setuptools wheel ninja
!python -m pip install -q nerfstudio gsplat huggingface_hub pillow plyfile
!rm -rf /content/video2scene_colab

In [ ]:
import os
import sys
import json
import shutil
import subprocess
from pathlib import Path
from PIL import Image
import torch
import gsplat
from huggingface_hub import snapshot_download, hf_hub_download

# --- CONFIGURATION ---
# If VIDEO_PATH is empty, the default 'poster' test example dataset will be downloaded.
VIDEO_PATH = "" # @param {type:"string"}
METHOD_NAME = "splatfacto-big" # @param ["splatfacto", "splatfacto-big"]
MAX_ITERATIONS = 15000 # @param {type:"integer"}

os.environ["MAX_JOBS"] = "1"
os.environ["CMAKE_BUILD_PARALLEL_LEVEL"] = "1"
os.environ["TORCH_CUDA_ARCH_LIST"] = "7.5"
os.environ["TORCH_EXTENSIONS_DIR"] = "/content/torch_extensions"
os.environ["TORCH_FORCE_NO_WEIGHTS_ONLY_LOAD"] = "1"

root = Path("/content/video2scene_colab")
root.mkdir(parents=True, exist_ok=True)

print(f"Using device: {torch.cuda.get_device_name(0)}")

In [ ]:
def extract_frames(video_input, output_dir, fps=5):
    """Extracts frames from a video file using ffmpeg."""
    output_dir.mkdir(parents=True, exist_ok=True)
    print(f"Extracting frames from {video_input} at {fps} FPS...")

    subprocess.run([
        "ffmpeg", "-i", str(video_input),
        "-vf", f"fps={fps}",
        "-q:v", "2",
        str(output_dir / "frame_%05d.png")
    ], check=True)

    return sorted(output_dir.glob("*.png"))

def prepare_data():
    data_dir = root / "data_source"
    images_dir = data_dir / "images"
    shutil.rmtree(data_dir, ignore_errors=True)
    images_dir.mkdir(parents=True)

    if VIDEO_PATH and os.path.exists(VIDEO_PATH):
        image_paths = extract_frames(Path(VIDEO_PATH), images_dir)
        print(f"Processed {len(image_paths)} frames from user video.")
    else:
        print("No video provided. Downloading test dataset...")
        snapshot_dir = root / "hf_snapshot"
        snapshot_download(
            repo_id="nerfstudioteam/datasets",
            repo_type="dataset",
            allow_patterns=["poster/images_2/*.png"],
            local_dir=snapshot_dir,
            local_dir_use_symlinks=False,
        )
        source_images = sorted((snapshot_dir / "poster" / "images_2").glob("*.png"))
        for img in source_images:
            shutil.copy2(img, images_dir / img.name)

        t_path = hf_hub_download(repo_id="nerfstudioteam/datasets", repo_type="dataset", filename="poster/transforms.json")
        pc_path = hf_hub_download(repo_id="nerfstudioteam/datasets", repo_type="dataset", filename="poster/sparse_pc.ply")

        shutil.copy2(pc_path, data_dir / "sparse_pc.ply")
        meta = json.loads(Path(t_path).read_text())

        with Image.open(source_images[0]) as img:
            w, h = img.size
        meta["w"], meta["h"] = w, h
        meta["ply_file_path"] = "sparse_pc.ply"
        (data_dir / "transforms.json").write_text(json.dumps(meta, indent=2))

    return data_dir

working_data = prepare_data()

In [ ]:
output_dir = root / "outputs"
log_path = root / "train.log"
shutil.rmtree(output_dir, ignore_errors=True)

print("Starting Training...")
!ns-train {METHOD_NAME} \
  --output-dir "{output_dir}" \
  --experiment-name experiment_1 \
  --max-num-iterations {MAX_ITERATIONS} \
  --vis tensorboard \
  --data "{working_data}" \
  > "{log_path}" 2>&1

print(f"Training finished. Log saved to {log_path}")

In [ ]:
config_files = sorted(output_dir.glob("experiment_1/*/*/config.yml"))
if config_files:
    config_path = config_files[-1]
    export_dir = root / "exports" / "splat_output"

    !ns-export gaussian-splat \
      --load-config "{config_path}" \
      --output-dir "{export_dir}"

    print(f"Exported to: {export_dir}")